# Notebook: Introduction to NLP in Python

Learn the basics of **Natural Language Processing (NLP)** step by step: tokenization, POS tagging, lemmatization, regular expressions and embeddings. Please also look at the **slides**!

Tip: run the cells from top to bottom. After each topic there is a small exercise.

Source: https://web.stanford.edu/~jurafsky/slp3/ed3book.pdf

### 0. Setup

We use **NLTK** and **spaCy**, two popular NLP libraries. First, we download the NLTK models we need.

In [ ]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("averaged_perceptron_tagger_eng")

Our example data is a table, so we first need a quick look at **pandas** (you have not used it yet). 

- A **DataFrame** is a table with rows and columns, like in Excel.
- `pd.read_csv("file.csv")` loads a CSV file into a DataFrame.
- `df.head()` shows the first 5 rows.
- `df["column"]` selects one column.
- `df["column"].head(3)` shows the first 3 values of that column.

Our example data is `tweets.csv`: tweets that mention German politicians in 2021, labeled with a sentiment (`POSITIVE`, `NEGATIVE`, `NEUTRAL`).

The file is uploaded on the **e-learning platform**. Download it and put it in the **same folder as this notebook**. Then we can load it.

In [ ]:
import pandas as pd

df = pd.read_csv("tweets.csv")
df.head()

`df["tweet"]` selects one column. Here are the first 3 tweets:

In [ ]:
for tweet in df["tweet"].head(3):
    print(tweet)
    print()

### 1. Tokenization

**Tokenization** splits a text into small units (tokens), e.g. words. You already know `split()` from the Python notebook:

In [ ]:
text = "Hello World! Python is a pretty cool programming language. I hope you like it too!"
print(text.split())

Problem: `"World!"` is one token, the punctuation is stuck to the word. A real **tokenizer** handles this.

In [ ]:
from nltk.tokenize import word_tokenize

print(word_tokenize(text))

NLTK can also split a text into **sentences**.

In [ ]:
print(nltk.sent_tokenize(text))

### Exercise 1: Tokenization

Print the tokens (as a list) of the first 3 tweets from `df`. Use `word_tokenize`.

In [ ]:
# Your solution here:

<details>
<summary><b>Show Solution</b></summary>

```python
for tweet in df["tweet"].head(3):
    print(word_tokenize(tweet))
```

</details>

### 2. Part-of-Speech (POS) Tagging

POS tagging finds the grammatical role of each word (noun, verb, ...). First with NLTK:

In [ ]:
text_en = "The quick brown fox jumps over the lazy dog."
tokens_en = word_tokenize(text_en)

print(nltk.pos_tag(tokens_en))

Now with **spaCy**, a very popular library with pre-trained models. First we load the English model and process the text.

In [ ]:
import spacy

nlp_en = spacy.load("en_core_web_sm")
doc_en = nlp_en(text_en)

Every token in the `doc` has attributes, e.g. `text` and `pos_`.

In [ ]:
for token in doc_en:
    print(f"{token.text} {token.pos_}")

There are also models for other languages, e.g. German.

In [ ]:
nlp_de = spacy.load("de_core_news_sm")
doc_de = nlp_de("Der schnelle braune Fuchs springt über den faulen Hund.")

for token in doc_de:
    print(f"{token.text} {token.pos_}")

### Exercise 2: POS Tagging

Print the POS tags of the first 3 tweets as a list of `(token, pos)` tuples. The tweets are German, so use `nlp_de`.

Example: `"Wir haben Pizzen gegessen."` -> `[("Wir", "PRON"), ("haben", "AUX"), ("Pizzen", "NOUN"), ("gegessen", "VERB"), (".", "PUNCT")]`

In [ ]:
# Your solution here:

<details>
<summary><b>Show Solution</b></summary>

```python
for tweet in df["tweet"].head(3):
    doc = nlp_de(tweet)
    print([(token.text, token.pos_) for token in doc])
```

</details>

### 3. Lemmatization and Stemming

One word can have many forms: "running", "ran", "runs" are all forms of "run". **Lemmatization** and **stemming** reduce words to a common base.

**Lemmatization** returns the real base form (the *lemma*). It uses part of speech and context, so `ate` becomes `eat`.

In [ ]:
text = "The striped bats were hanging on their feet and ate best fishes"
doc = nlp_en(text)

for token in doc:
    print(f"{token.text} → {token.lemma_}")

**Stemming** simply cuts off word endings with fixed rules. It is faster, but the result is not always a real word. We use the well-known Porter Stemmer.

In [ ]:
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()

for word in text.split():
    print(f"{word} → {stemmer.stem(word)}")

### Exercise 3: Lemmatization

Print the lemmas of the first 3 tweets as a list of `(token, lemma)` tuples (use `nlp_de`).

Example: `"Wir haben Pizzen gegessen."` -> `[("Wir", "wir"), ("haben", "haben"), ("Pizzen", "Pizza"), ("gegessen", "essen"), (".", ".")]`

In [ ]:
# Your solution here:

<details>
<summary><b>Show Solution</b></summary>

```python
for tweet in df["tweet"].head(3):
    doc = nlp_de(tweet)
    print([(token.text, token.lemma_) for token in doc])
```

</details>

### 4. Regular Expressions (RegEx)

A **regular expression** describes a text pattern. With it we can find and extract things like numbers, emails or dates.

Tip: test your patterns on [RegExr.com](https://regexr.com/) and use a [cheatsheet](https://web.mit.edu/hackl/www/lab/turkshop/slides/regex-cheatsheet.pdf).

`re.search()` checks if a pattern occurs. `\b` marks a word boundary. The `r` before the string means "raw string" (backslashes are kept as they are).

In [ ]:
import re

text = "You still have 45GB of storage and your email is mi@ur.de and you have 3 files with 100MB, 250MB and 1GB."

print(bool(re.search(r"\bstorage\b", text)))

`re.findall()` returns **all** matches. `\d` is a digit, `+` means "one or more".

In [ ]:
print(re.findall(r"\d+", text))

Some useful symbols:

| Pattern | Meaning |
|---|---|
| `\d` | a digit |
| `\w` | a letter, digit or `_` |
| `\s` | a whitespace |
| `+` | one or more |
| `*` | zero or more |
| `?` | optional (zero or one) |
| `{2,4}` | 2 to 4 times |
| `[abc]` | one of the characters a, b, c |
| `(?:A\|B)` | A or B |

### Exercise 4: Hashtags

Find all hashtags in the text (e.g. `#Python3`).

In [ ]:
text = "Loving the new features in #Python3! #programming #NLP is fun. Visit us at https://example.com #AI"

# Your solution here:

<details>
<summary><b>Show Solution</b></summary>

```python
hashtags = re.findall(r"#\w+", text)
print(hashtags)
```

</details>

### Exercise 5: Words ending with "ing"

Find all words that end with `ing`.

In [ ]:
text = "I am enjoying swimming and reading while running in the morning."

# Your solution here:

<details>
<summary><b>Show Solution</b></summary>

```python
ing_words = re.findall(r"\b\w+ing\b", text)
print(ing_words)
```

</details>

Back to our storage text. We want values like `45GB` or `100MB`: a number, optional spaces, then a unit.

In [ ]:
text = "You still have 45GB of storage and your email is mi@ur.de and you have 3 files with 100MB, 250MB and 1GB."

print(re.findall(r"\b\d+\s*(?:GB|MB|KB)\b", text))

Why `(?:...)`? With a normal group `(GB|MB|KB)`, `findall()` would return *only* the group (`"GB"`) and drop the number. `(?:...)` groups without capturing.

Emails can be found with a pattern, too:

In [ ]:
print(re.findall(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}", text))

Complex formats (emails, phone numbers) have many special cases. For these, **use a library** instead of your own pattern, e.g. `email_validator` (checks the format and the domain).

In [ ]:
from email_validator import validate_email, EmailNotValidError

try:
    result = validate_email("mi@ur.de")
    print("Valid email:", result.email)
except EmailNotValidError as e:
    print("Invalid email:", e)

### Exercise 6: The name Maier

Find all spellings of the last name "Maier".

In [ ]:
text = "Meyer, Maier, Meier, Mair, Meir, Mayr, Meyr"

# Your solution here:

<details>
<summary><b>Show Solution</b></summary>

```python
variants = re.findall(r"\bM[ae][iy]?[ae]?[iy]?r\b", text)
print(variants)
```

</details>

### Exercise 7: Dates

Find all dates in the format `Day. Month Year`, e.g. `1. January 1901`.

In [ ]:
text = "1. January 1901 29. February 2099, 02. March 1234, 1. October 2015"

# Your solution here:

<details>
<summary><b>Show Solution</b></summary>

```python
months = "January|February|March|April|May|June|July|August|September|October|November|December"
dates = re.findall(rf"\b\d{{1,2}}\. (?:{months}) \d{{4}}\b", text)
print(dates)
```

</details>

### 5. More Pandas

Text data often comes as a table. We already loaded `tweets.csv`; here are a few more useful `pandas` tricks.

`df["sentiment"] == "POSITIVE"` creates a True/False filter. With it we select only some rows.

In [ ]:
df[df["sentiment"] == "POSITIVE"].head()

`value_counts()` counts how often each value occurs. Positive tweets are by far the rarest class.

In [ ]:
df["sentiment"].value_counts()

`apply()` runs a function on every value of a column. Here we count the words of each tweet and store them in a new column.

In [ ]:
def word_count(text):
    return len(text.split())

df["word_count"] = df["tweet"].apply(word_count)
df[["tweet", "word_count"]].head()

`sort_values()` sorts the table by a column.

In [ ]:
df.sort_values(by="word_count").head()

`groupby()` groups the rows, then we can compute a value per group, e.g. the average word count per sentiment.

In [ ]:
df.groupby("sentiment")["word_count"].mean()

`apply()` also works for filters. Which tweets mention "Berlin"?

In [ ]:
def has_berlin(text):
    return "Berlin" in text

df[df["tweet"].apply(has_berlin)].head(3)

### Exercise 8: Pandas

1. How many tweets does each party (`source_party`) have?
2. What is the average word count of the tweets per party?

In [ ]:
# Your solution here:

<details>
<summary><b>Show Solution</b></summary>

```python
print(df["source_party"].value_counts())
print(df.groupby("source_party")["word_count"].mean())
```

</details>

### 6. NumPy Basics

**NumPy** is the library for fast number crunching with arrays. We need it to understand embeddings in the next section.

`np.array()` turns a list into an array. `shape` tells its size.

In [ ]:
import numpy as np

arr = np.array([1, 2, 3, 4, 5])
print(arr)
print(arr.shape)

A 2D array is a matrix. `(3, 3)` means 3 rows and 3 columns.

In [ ]:
matrix = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
print(matrix)
print(matrix.shape)

Indexing works like with lists. For 2D arrays you give `[row, column]`; `:` means "all".

In [ ]:
print(matrix[0, 0])   # first row, first column
print(matrix[1, 2])   # second row, third column
print(matrix[0, :])   # whole first row
print(matrix[:, 0])   # whole first column

Calculations work **element by element**, without a loop.

In [ ]:
arr1 = np.array([1, 2, 3, 4])
arr2 = np.array([10, 20, 30, 40])

print(arr1 + arr2)
print(arr1 * 2)

Useful functions for statistics:

In [ ]:
data = np.array([1, 5, 2, 8, 3, 7, 4, 6])

print("Sum:", np.sum(data))
print("Mean:", np.mean(data))
print("Max:", np.max(data), "at index", np.argmax(data))
print("Sorted:", np.sort(data))

`reshape()` changes the shape. `-1` means: NumPy figures out this size itself.

In [ ]:
arr = np.array([1, 2, 3, 4, 5, 6])

print(arr.reshape(2, 3))
print(arr.reshape(-1, 2))

### Exercise 9: NumPy

Given is `arr = np.array([4, 8, 15, 16, 23, 42])`.
1. Print the mean.
2. Multiply all values by 10.
3. Reshape the array into 3 rows and 2 columns.

In [ ]:
# Your solution here:

<details>
<summary><b>Show Solution</b></summary>

```python
arr = np.array([4, 8, 15, 16, 23, 42])

print(np.mean(arr))
print(arr * 10)
print(arr.reshape(3, 2))
```

</details>

### 7. Embeddings and Similarity

An **embedding** represents a word as a vector (a list of numbers). Words with a similar meaning get similar vectors. With spaCy, the `.vector` attribute returns the embedding.

In [ ]:
vec_cat = nlp_en("cat").vector

print(vec_cat.shape)
print(vec_cat[:5])

To compare two vectors we use the **cosine similarity**: close to 1 means very similar, close to 0 means unrelated.

`cosine_similarity` expects 2D arrays, so we turn each vector into one row with `reshape(1, -1)`.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def similarity(word1, word2):
    vec1 = nlp_en(word1).vector.reshape(1, -1)
    vec2 = nlp_en(word2).vector.reshape(1, -1)
    return cosine_similarity(vec1, vec2)[0][0]

print(similarity("cat", "dog"))

Now we compare several pairs: two similar and two unrelated ones.

In [ ]:
word_pairs = [
    ("cat", "dog"),
    ("car", "bike"),
    ("cheesecake", "swimming"),
    ("death", "google"),
]

for w1, w2 in word_pairs:
    print(f"'{w1}' vs. '{w2}': {similarity(w1, w2):.3f}")

### Exercise 10: Similarity

Compare `"king"` with `"queen"` and with `"banana"` using `similarity()`. Which word is more similar to `"king"`?

In [ ]:
# Your solution here:

<details>
<summary><b>Show Solution</b></summary>

```python
print(similarity("king", "queen"))
print(similarity("king", "banana"))
```

</details>